# HaluRISC — Full Training Pipeline (Colab) — corrected grouped-split protocol

Runs the complete experiment protocol on a Colab GPU (T4 or better):

1. HaluEval download + prepare with a **GROUP-AWARE 70/15/15 split** (both answers of one question stay in the same partition; a leakage report is generated and asserted)
2. Full feature extraction (length, lexical, entity/NER, **NLI**, numeric, hedging, semantic) + NLI checkpoint provenance
3. XGBoost tuning (30 iters, 5-fold CV) + baselines + 3-seed protocol + early stopping
4. Platt vs isotonic calibration, ECE/Brier, McNemar, bootstrap CIs, Wilcoxon, 7-group ablations
5. SHAP global + local explanations
6. RAGTruth zero-shot external validation
7. Error analysis (10 FP + 10 FN, auto-tagged for manual review)
8. Latency/efficiency analysis
9. Optional LLM-as-judge comparison (needs OPENAI_API_KEY)
10. Artifact manifest generation (hashes, versions, hardware, split report)
11. Zip artifacts + feature matrix to **Google Drive** for download

**Before starting:** have `colab/halurisc_src.zip` from the repo ready. Cell 3 opens a file-picker to select it from your laptop (no Drive upload needed). Alternatively, upload it once to `MyDrive/HaluRISC/halurisc_src.zip` and it will be picked up automatically.

**Runtime:** enable GPU (Runtime > Change runtime type). **Use T4 (16 GB) or L4 (24 GB) if available — an A100 is overkill for this workload** (the models are small by design; total VRAM use is only ~2 GB). Total wall time ≈ 15-30 min.

**After the run:** download the zip, unzip at the repo root of your laptop, then start the API: `& .venv\Scripts\python.exe -m uvicorn src.api.main:app --port 8000`.

In [ ]:
# 1) Mount Google Drive (artifacts persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/HaluRISC'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted at', DRIVE_DIR)

In [ ]:
# 2) Environment check: GPU must be enabled
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('!! No GPU detected - enable GPU in Runtime > Change runtime type')
    raise SystemExit(1)

In [ ]:
# 3) Get the HaluRISC source. PREFERRED: browser file-picker (select colab/halurisc_src.zip from your laptop).
#    Fallback: if you already put halurisc_src.zip inside Drive/HaluRISC/, it is used automatically.
import zipfile, io, os
from google.colab import files

ROOT = '/content/HaluRISC'
os.makedirs(ROOT, exist_ok=True)

if not os.path.exists(os.path.join(ROOT, 'src')):
    src_zip = None
    drive_zip = os.path.join(DRIVE_DIR, 'halurisc_src.zip')
    if os.path.exists(drive_zip):
        src_zip = open(drive_zip, 'rb')
        print('Found halurisc_src.zip on Drive - extracting...')
        with zipfile.ZipFile(src_zip) as z:
            z.extractall(ROOT)
    else:
        print('Upload halurisc_src.zip (repo/colab/halurisc_src.zip) - use the file picker:')
        uploaded = files.upload()
        for name, content in uploaded.items():
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                z.extractall(ROOT)
    print('Extracted to', ROOT)
print('src present:', os.path.exists(os.path.join(ROOT, 'src')))
%cd {ROOT}

In [ ]:
# 4) Install pinned dependencies (Colab keeps its own torch)
!pip install -q -r colab/requirements-colab.txt
!python -m spacy download en_core_web_sm -q
print('deps OK')

In [ ]:
# 5) HaluEval download + prepare with GROUP-AWARE split (item_idx) + integrity check
!python src/data/download.py
!python src/data/prepare.py
import json
rep = json.load(open('artifacts/split_integrity_report.json'))
print(json.dumps(rep, indent=2))
assert rep['leakage_free'] and rep['groups_spanning_multiple_splits'] == 0, 'Split leakage detected!'

In [ ]:
# 6) Full feature extraction (7 groups, ~40K NLI pairs on GPU; ~5-10 min on T4/L4)
# --device cuda -> models load in fp16 + CUDA; --batch-size 256 -> bigger GPU batches = faster
# Default NLI model: cross-encoder/nli-deberta-v3-base. Fallback: --nli-model cross-encoder/nli-MiniLM2-L6-H768
!python src/features/extract_features.py --device cuda --batch-size 256
import json
print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))

In [ ]:
# 7) Full experiment protocol: tuning, baselines, 3 seeds, early stopping, calibration, stats, ablations (~10-20 min)
# HALU_XGB_DEVICE=cpu: xgb training is tiny (14K x 26); CPU keeps the saved models
# portable so they load on any machine (CUDA-trained boosters do NOT port across platforms).
import os
os.environ['HALU_XGB_DEVICE'] = 'cpu'
!python src/models/train_pipeline.py

In [ ]:
# 8) SHAP explanations + calibration/ROC/PR figures
!python src/explain/shap_analysis.py

In [ ]:
# 9) RAGTruth zero-shot external validation (downloads from HuggingFace)
!python src/data/download_ragtruth.py
!python src/models/eval_ragtruth.py

In [ ]:
# 10) Error analysis (10 FP + 10 FN, auto-tagged with the blueprint taxonomy)
# NOTE: categories are heuristic and must be manually reviewed in
# artifacts/results/error_analysis_cases.json before the paper uses them.
!python src/models/error_analysis.py

In [ ]:
# 11) Latency / efficiency analysis (features + predict + SHAP, p50/p95)
!python src/models/eval_efficiency.py

In [ ]:
# 12) OPTIONAL: LLM-as-judge comparison (200 samples, ~$0.05-0.15)
# Requires OPENAI_API_KEY. In Colab: Edit > Notebook settings > Secrets, add OPENAI_API_KEY.
import os
if os.environ.get('OPENAI_API_KEY'):
    os.system('python src/models/eval_llm_judge.py')
else:
    print('OPENAI_API_KEY not set - skipping optional LLM-as-judge. You can run it later locally.')

In [ ]:
# 13) Generate the artifact manifest (hashes, versions, hardware, split report)
!python src/models/make_manifest.py
import json
m = json.load(open('artifacts/results/manifest.json'))
print(json.dumps({k: m[k] for k in ['generated_at', 'git_commit', 'model_version', 'nli_model', 'split_report']}, indent=2))

In [ ]:
# 14) Show the headline results (model comparison, calibration, RAGTruth, per-seed rows)
import json, pandas as pd
res = json.load(open('artifacts/results/final_results.json'))
rows = {k: v for k, v in res.items() if isinstance(v, dict) and 'f1' in v and 'auroc' in v}
df = pd.DataFrame(rows).T[['precision','recall','f1','auroc','pr_auc','mcc']].round(4)
print('MODEL COMPARISON (test set, mean over seeds 42/123/456)')
print(df.to_string())
print('\nCALIBRATION:')
print(json.dumps(res['calibration'], indent=2))
print('\nABLATION:')
print(pd.DataFrame(res['ablation']).to_string(index=False))
print('\nPER-SEED XGBOOST:')
print(pd.DataFrame(json.load(open('artifacts/results/seed_metrics.json'))['xgboost']).round(4).to_string(index=False))
rt = json.load(open('artifacts/results/ragtruth_results.json'))
print('\nRAGTRUTH ZERO-SHOT: f1=%.4f auroc=%.4f ece=%.4f (n=%d)' % (rt['f1'], rt['auroc'], rt['ece'], rt['n_samples']))

In [ ]:
# 15) Package artifacts to Drive (persists across sessions) and offer a download link
import os, zipfile
from datetime import date
from google.colab import files

stamp = date.today().isoformat()
zip_path = f'{DRIVE_DIR}/halurisc_artifacts_{stamp}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk('artifacts'):
        for fn in fnames:
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, '.'))
    for fn in ['data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
               'data/processed/nli_model_used.json', 'data/processed/audit_50_samples.json']:
        if os.path.exists(fn):
            z.write(fn)
print('Saved:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
print()
print('NEXT: download the zip from your Drive, unzip at the repo root of your laptop.')
print('The API (uvicorn) and web dashboard will then load the corrected artifacts.')
files.download(zip_path)